# TDSal Camera-Ready — Reviewer Fixes Notebook (Local)

Run this notebook top-to-bottom to address every reviewer code/experiment checklist item.

Checklist coverage:
1. SBERT inference-mode bug fix (`emb = emb.detach().clone()`) in `TaskEncoder.forward()`
2. `compute_efficiency.py` equivalent: trainable params, FLOPs, CPU/GPU inference time
3. `verify_backbone_shapes.py` equivalent: assert YOLO backbone output `[B, 512, 12, 12]` for `384x384`
4. AUC-Borji sign fix (`auc = -trapz(tp, fp)`) and ablation rerun with corrected metric
5. `evaluate_final.py` equivalent: final test metrics compared against the paper reference values

Local setup is handled in the next cells (absolute paths + package install).


In [ ]:
import os

# Update these three paths before running if your local layout differs.
DATA_PATH     = '/Users/canmizrakli/non-icloud-storage/CMPE490/TDSP.Net/Task-based-eye-fixation-dataset_1024x768'
CKPT_PATH     = '/Users/canmizrakli/non-icloud-storage/CMPE490/TDSP.Net/models/tdsp_40epoch.pth'
ABLATION_BASE = '/Users/canmizrakli/non-icloud-storage/CMPE490/TDSP.Net/models'

print('DATA_PATH =', DATA_PATH)
print('CKPT_PATH =', CKPT_PATH)
print('ABLATION_BASE =', ABLATION_BASE)

if not os.path.isdir(DATA_PATH):
    raise FileNotFoundError(f'Dataset not found: {DATA_PATH}')
if not os.path.exists(CKPT_PATH):
    raise FileNotFoundError(f'Checkpoint not found: {CKPT_PATH}')
if not os.path.isdir(ABLATION_BASE):
    print(f'WARNING: ablation directory not found: {ABLATION_BASE}')


In [ ]:
# Local install (safe to rerun).
%pip install --quiet torch torchvision einops ultralytics thop pandas matplotlib 'sentence-transformers<4.0.0'


## 2. Imports

In [ ]:
import os, glob, random, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
import torchvision.transforms.functional as TF
from einops import rearrange
from PIL import Image
from torch.utils.data import Dataset, DataLoader, random_split
from ultralytics import YOLO
from sentence_transformers import SentenceTransformer
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
NUM_WORKERS = min(2, os.cpu_count() or 1)
print('Device:', device)
print('DataLoader workers:', NUM_WORKERS)

def safe_torch_load(path, map_location='cpu'):
    try:
        return torch.load(path, map_location=map_location, weights_only=True)
    except TypeError:
        # For older torch versions without weights_only arg.
        return torch.load(path, map_location=map_location)


## 3. Data Augmentation Helpers

In [ ]:
class PairedRandomHorizontalFlip:
    def __init__(self, p=0.5): self.p = p
    def __call__(self, img, sal):
        if random.random() < self.p:
            img = TF.hflip(img); sal = TF.hflip(sal)
        return img, sal

class PairedRandomRotation:
    def __init__(self, degrees=10): self.degrees = degrees
    def __call__(self, img, sal):
        angle = random.uniform(-self.degrees, self.degrees)
        return TF.rotate(img, angle), TF.rotate(sal, angle)

## 4. Dataset

In [ ]:
task_mapping = {
    'task1': 'free view',
    'task2': 'count people',
    'task3': 'detect the emotion',
    'task4': 'identify the action',
}

class TaskSaliencyDataset(Dataset):
    def __init__(self, data_root, task_mapping, transform=None,
                 saliency_transform=None, paired_transforms=None):
        self.data_root = data_root
        self.task_mapping = task_mapping
        self.transform = transform
        self.saliency_transform = saliency_transform
        self.paired_transforms = paired_transforms
        self.tasks = list(task_mapping.keys())
        self.samples = []
        for task in self.tasks:
            fdm_folder = os.path.join(data_root, task, 'fdm')
            for fdm_file in glob.glob(os.path.join(fdm_folder, '*.png')):
                base = os.path.splitext(os.path.basename(fdm_file))[0]
                for ext in ('.jpg', '.png'):
                    sp = os.path.join(data_root, 'stimuli', base + ext)
                    if os.path.exists(sp):
                        self.samples.append((sp, fdm_file, task))
                        break

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        stimuli_path, fdm_path, task = self.samples[idx]
        img = Image.open(stimuli_path).convert('RGB')
        fdm = Image.open(fdm_path).convert('L')
        if self.transform:          img = self.transform(img)
        if self.saliency_transform: fdm = self.saliency_transform(fdm)
        else:                       fdm = T.ToTensor()(fdm)
        if self.paired_transforms:
            for t in self.paired_transforms:
                img, fdm = t(img, fdm)
        return {'stimuli': img, 'fdm': fdm,
                'task': task,
                'task_description': self.task_mapping[task]}

# ── Load dataset ─────────────────────────────────────────────────────────
img_tf     = T.Compose([T.Resize((384, 384)), T.ToTensor()])
sal_tf     = T.Compose([T.Resize((384, 384)), T.ToTensor()])

dataset = TaskSaliencyDataset(
    DATA_PATH, task_mapping,
    transform=img_tf, saliency_transform=sal_tf,
    paired_transforms=None  # no augmentation at inference
)
print(f'Total samples: {len(dataset)}')

## 5. Reproduce Test Split (seed=42, 70/15/15)

In [ ]:
total_size = len(dataset)
train_size = int(0.70 * total_size)
val_size   = int(0.15 * total_size)
test_size  = total_size - train_size - val_size

generator = torch.Generator().manual_seed(42)
train_ds, val_ds, test_ds = random_split(
    dataset, [train_size, val_size, test_size], generator=generator
)
test_loader = DataLoader(test_ds, batch_size=8, shuffle=False, num_workers=NUM_WORKERS)
print(f'Train / Val / Test: {len(train_ds)} / {len(val_ds)} / {len(test_ds)}')


## 6. Model Definitions

`TaskEncoder` below includes the SBERT inference-mode fix (`detach().clone()`) requested by reviewers.


In [ ]:
class YOLOBackbone(nn.Module):
    """YOLOv5su layers 0-9 (truncated at SPPF). Output: [B, 512, H/32, W/32]."""
    def __init__(self, model_name='yolov5su.pt', cut_layer=10):
        super().__init__()
        yolo = YOLO(model_name)
        self.feature_extractor = yolo.model.model[:cut_layer]
        for p in self.feature_extractor.parameters():
            p.requires_grad = False
        # Reviewer clarification: exactly one pre-FPM feature map, no multiscale fusion.
        self.num_pre_fpm_feature_maps = 1
        self.num_multiscale_fusions_before_fpm = 0
    def get_pre_fpm_feature_info(self):
        return {
            'num_pre_fpm_feature_maps': self.num_pre_fpm_feature_maps,
            'num_multiscale_fusions_before_fpm': self.num_multiscale_fusions_before_fpm,
        }
    def forward(self, x):
        return self.feature_extractor(x).clone()

class SimpleFPN(nn.Module):
    def __init__(self, in_channels=512, out_channels=128):
        super().__init__()
        self.conv_out = nn.Conv2d(in_channels, out_channels, kernel_size=1)
    def forward(self, x): return self.conv_out(x)

class TaskEncoder(nn.Module):
    """SBERT + linear projection.  detach().clone() fixes inference-mode tensor crash."""
    def __init__(self, output_dim=64):
        super().__init__()
        self.text_encoder = SentenceTransformer('all-MiniLM-L6-v2')
        for p in self.text_encoder.parameters():
            p.requires_grad = False
        self.linear = nn.Linear(384, output_dim)
    def forward(self, task_descriptions):
        dev = next(self.linear.parameters()).device
        emb = self.text_encoder.encode(task_descriptions, convert_to_tensor=True)
        emb = emb.detach().clone().to(dev)   # fix: exit inference_mode graph + ensure device match
        return F.relu(self.linear(emb))

class TransformerFusion(nn.Module):
    def __init__(self, d_model=128, nhead=4, num_layers=1, task_embed_dim=64):
        super().__init__()
        self.query_proj = nn.Linear(task_embed_dim, d_model)
        enc_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, batch_first=False)
        self.transformer_encoder = nn.TransformerEncoder(
            enc_layer, num_layers=num_layers, enable_nested_tensor=False)
    def forward(self, vision_feats, task_embed):
        B, C, H, W = vision_feats.shape
        v_seq = rearrange(vision_feats, 'b c h w -> (h w) b c')
        t_q   = rearrange(self.query_proj(task_embed), 'b d -> 1 b d')
        enc   = self.transformer_encoder(torch.cat([t_q, v_seq], 0))
        return rearrange(enc[1:], '(h w) b c -> b c h w', h=H, w=W)

class SaliencyDecoder(nn.Module):
    def __init__(self, in_channels=128):
        super().__init__()
        self.conv1   = nn.Conv2d(in_channels, 64, 3, padding=1)
        self.deconv1 = nn.ConvTranspose2d(64, 32, 4, stride=2, padding=1)
        self.deconv2 = nn.ConvTranspose2d(32,  1, 4, stride=2, padding=1)
    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.deconv1(x))
        return torch.sigmoid(self.deconv2(x))

class YOLOTaskSaliencyModel(nn.Module):
    def __init__(self, task_embed_dim=64, vision_dim=128, nhead=4, num_layers=1):
        super().__init__()
        self.backbone           = YOLOBackbone(model_name='yolov5su.pt')
        self.fpn                = SimpleFPN(512, vision_dim)
        self.task_encoder       = TaskEncoder(task_embed_dim)
        self.transformer_fusion = TransformerFusion(vision_dim, nhead, num_layers, task_embed_dim)
        self.saliency_decoder   = SaliencyDecoder(vision_dim)
    def forward(self, images, task_descriptions):
        feat   = self.fpn(self.backbone(images))
        t_emb  = self.task_encoder(task_descriptions)
        fused  = self.transformer_fusion(feat, t_emb)
        return self.saliency_decoder(fused)

## 7. Load Checkpoint

In [ ]:
if not os.path.exists(CKPT_PATH):
    raise FileNotFoundError(f'Checkpoint not found: {CKPT_PATH}')

model = YOLOTaskSaliencyModel()
state = safe_torch_load(CKPT_PATH, map_location=device)
model.load_state_dict(state, strict=False)
model.to(device).eval()
print('Checkpoint loaded:', CKPT_PATH)


---
## Section A — Efficiency Metrics (`compute_efficiency.py` equivalent)


In [ ]:
def count_parameters(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)

def measure_inference_time(m, imgs, tasks, dev, n_runs=100, warmup=10):
    m.eval(); m.to(dev); imgs = imgs.to(dev)
    with torch.no_grad():
        for _ in range(warmup): m(imgs, tasks)
    times = []
    with torch.no_grad():
        for _ in range(n_runs):
            if dev.type == 'cuda': torch.cuda.synchronize()
            t0 = time.perf_counter()
            m(imgs, tasks)
            if dev.type == 'cuda': torch.cuda.synchronize()
            times.append((time.perf_counter() - t0) * 1000)
    return np.mean(times), np.std(times)

dummy_imgs  = torch.randn(1, 3, 384, 384)
dummy_tasks = ['free view']

n_params = count_parameters(model)
print(f"{'='*55}")
print(f"Trainable parameters : {n_params:,}  ({n_params/1e6:.2f} M)")

try:
    import copy
    from thop import profile, clever_format
    class _Wrap(nn.Module):
        def __init__(self, m): super().__init__(); self.m = m
        def forward(self, x): return self.m(x, ['free view'])
    _model_cpu = copy.deepcopy(model).cpu()
    flops, _ = profile(_Wrap(_model_cpu), inputs=(dummy_imgs,), verbose=False)
    del _model_cpu
    flops_str, _ = clever_format([flops, n_params], '%.3f')
    print(f"FLOPs (single image) : {flops_str}")
except Exception as e:
    print(f'FLOPs computation skipped: {e}')

cpu_model = YOLOTaskSaliencyModel()
cpu_model.load_state_dict(safe_torch_load(CKPT_PATH, map_location='cpu'), strict=False)
mean_cpu, std_cpu = measure_inference_time(
    cpu_model, dummy_imgs, dummy_tasks, torch.device('cpu')
)
print(f"Inference time (CPU) : {mean_cpu:.1f} ± {std_cpu:.1f} ms  (n=100)")

if torch.cuda.is_available():
    mean_gpu, std_gpu = measure_inference_time(
        model, dummy_imgs, dummy_tasks, device
    )
    print(f"Inference time (GPU) : {mean_gpu:.1f} ± {std_gpu:.1f} ms  (n=100)")
else:
    print('Inference time (GPU) : no CUDA available on this runtime')

print(f"{'='*55}")
print()
print('LaTeX snippet for Section 3.2:')
print(f'TDSal comprises approximately \\textbf{{{n_params/1e6:.1f}~M}} trainable parameters')
print(f'and achieves a mean inference time of \\textbf{{{mean_cpu:.0f}~ms}} on CPU (n=100).')

---
## Section B — Backbone Shape Verification (`verify_backbone_shapes.py` equivalent)


In [ ]:
with torch.no_grad():
    _dummy = torch.randn(1, 3, 384, 384).to(device)
    _bb_out = model.backbone(_dummy)
    _fpn_out = model.fpn(_bb_out)

n_layers = len(list(model.backbone.feature_extractor))

print('='*50)
print(f'Input shape         : {tuple(_dummy.shape)}')
print(f'Backbone output     : {tuple(_bb_out.shape)}  <- single scale, stride-32')
print(f'FPM output          : {tuple(_fpn_out.shape)}  <- 512->128 ch reduction')
print(f'Backbone layers     : {n_layers}  (layers 0–{n_layers-1})')
print(f'Multi-scale outputs : 0  (single-scale only)')
print(f'Pre-FPM YOLO maps  : {model.backbone.num_pre_fpm_feature_maps}')
print(f'MS fusions before FPM: {model.backbone.num_multiscale_fusions_before_fpm}')
print('='*50)

assert _bb_out.shape  == (1, 512, 12, 12), 'Unexpected backbone output shape!'
assert _fpn_out.shape == (1, 128, 12, 12), 'Unexpected FPM output shape!'
print('Assertions passed.')

print()
print('LaTeX snippet for Section 3.2.1:')
print('The YOLOv5su backbone is truncated at layer~9 (SPPF), retaining layers~0--9')
print('and discarding the detection head. For a $384{\times}384$ input, this yields')
print('a single feature map of shape $[B,\,512,\,12,\,12]$ (stride-32). This')
print('single-scale output is passed directly to the FPM; no multi-scale pyramid')
print('is constructed prior to the FPM.')

---
## Section C — Corrected Final Evaluation (`evaluate_final.py` equivalent)


In [ ]:
EPS = 1e-8
_trapz = getattr(np, 'trapezoid', np.trapz)  # np.trapz removed in NumPy 2.0

def cc_metric(pred, gt):
    B = pred.shape[0]
    p = pred.view(B, -1) - pred.view(B, -1).mean(1, keepdim=True)
    g = gt.view(B, -1)   - gt.view(B, -1).mean(1, keepdim=True)
    return ((p * g).sum(1) / (torch.sqrt((p**2).sum(1) * (g**2).sum(1) + EPS))).mean().item()

def kl_metric(pred, gt):
    B = pred.shape[0]
    p = pred.view(B, -1).clamp(min=EPS); p = p / (p.sum(1, keepdim=True) + EPS)
    g = gt.view(B, -1).clamp(min=EPS);   g = g / (g.sum(1, keepdim=True) + EPS)
    return (g * (g.log() - p.log())).sum(1).mean().item()

def sim_metric(pred, gt):
    B = pred.shape[0]
    p = pred.view(B, -1).clamp(min=0); p = p / (p.sum(1, keepdim=True) + EPS)
    g = gt.view(B, -1).clamp(min=0);   g = g / (g.sum(1, keepdim=True) + EPS)
    return torch.min(p, g).sum(1).mean().item()

def nss_metric(pred, fix):
    B = pred.shape[0]
    s = pred.view(B, -1); f = fix.view(B, -1)
    s = (s - s.mean(1, keepdim=True)) / (s.std(1, keepdim=True) + EPS)
    return ((s * f).sum(1) / f.sum(1).clamp(min=1)).mean().item()

def auc_borji_metric(pred, fix, n_splits=100, step=0.1):
    """AUC-Borji in [0,1]. Negated trapz because fp/tp curve is traced right-to-left."""
    pred = pred.detach().cpu().numpy()
    fix  = fix.detach().cpu().numpy()
    aucs = []
    for b in range(pred.shape[0]):
        s_map = pred[b, 0]
        f_map = fix[b, 0].astype(bool)
        if f_map.sum() == 0: continue
        S       = (s_map - s_map.min()) / (s_map.max() - s_map.min() + EPS)
        S_fix   = S[f_map]
        neg_idx = np.where(~f_map.flatten())[0]
        if len(neg_idx) == 0: continue
        for _ in range(n_splits):
            S_rand = S.flatten()[
                np.random.choice(neg_idx, S_fix.size, replace=(len(neg_idx) < S_fix.size))
            ]
            th = np.arange(0, 1 + step, step)
            tp = np.array([(S_fix  >= t).mean() for t in th])
            fp = np.array([(S_rand >= t).mean() for t in th])
            aucs.append(-_trapz(tp, fp))
    return float(np.mean(aucs)) if aucs else float('nan')

def _roc_auc_from_scores(pos_scores, neg_scores):
    pos_scores = np.asarray(pos_scores).reshape(-1)
    neg_scores = np.asarray(neg_scores).reshape(-1)
    if pos_scores.size == 0 or neg_scores.size == 0:
        return float('nan')
    scores = np.concatenate([pos_scores, neg_scores]).astype(np.float64)
    labels = np.concatenate([
        np.ones(pos_scores.size, dtype=np.int32),
        np.zeros(neg_scores.size, dtype=np.int32)
    ])
    order = np.argsort(-scores, kind='mergesort')
    labels = labels[order]
    tp = np.cumsum(labels == 1) / (pos_scores.size + EPS)
    fp = np.cumsum(labels == 0) / (neg_scores.size + EPS)
    tp = np.concatenate([[0.0], tp, [1.0]])
    fp = np.concatenate([[0.0], fp, [1.0]])
    return float(_trapz(tp, fp))

def auc_judd_metric(pred, fix):
    pred = pred.detach().cpu().numpy()
    fix  = fix.detach().cpu().numpy()
    aucs = []
    for b in range(pred.shape[0]):
        s_map = pred[b, 0]
        f_map = fix[b, 0].astype(bool)
        if f_map.sum() == 0:
            continue
        S = (s_map - s_map.min()) / (s_map.max() - s_map.min() + EPS)
        aucs.append(_roc_auc_from_scores(S[f_map], S[~f_map]))
    return float(np.nanmean(aucs)) if aucs else float('nan')

def sauc_metric(pred, fix):
    pred = pred.detach().cpu().numpy()
    fix  = fix.detach().cpu().numpy().astype(bool)
    B = pred.shape[0]
    aucs = []
    for b in range(B):
        s_map = pred[b, 0]
        f_map = fix[b, 0]
        if f_map.sum() == 0:
            continue
        neg_mask = np.zeros_like(f_map, dtype=bool)
        for j in range(B):
            if j == b:
                continue
            neg_mask |= fix[j, 0]
        neg_mask &= ~f_map
        if neg_mask.sum() == 0:
            continue
        S = (s_map - s_map.min()) / (s_map.max() - s_map.min() + EPS)
        aucs.append(_roc_auc_from_scores(S[f_map], S[neg_mask]))
    return float(np.nanmean(aucs)) if aucs else float('nan')

def _append_if_finite(bucket, value):
    if np.isfinite(value):
        bucket.append(float(value))

def evaluate_saliency_model(m, loader, dev):
    m.to(dev).eval()
    scores = {k: [] for k in ['CC', 'KL', 'SIM', 'NSS', 'AUC-Borji', 'AUC-J', 'sAUC']}
    with torch.no_grad():
        for batch in loader:
            imgs  = batch['stimuli'].to(dev)
            gts   = batch['fdm'].to(dev)
            descs = batch['task_description']
            preds = m(imgs, descs)
            preds = F.interpolate(preds, gts.shape[-2:], mode='bilinear', align_corners=False)
            fix   = (gts > 0.5).float()
            _append_if_finite(scores['CC'], cc_metric(preds, gts))
            _append_if_finite(scores['KL'], kl_metric(preds, gts))
            _append_if_finite(scores['SIM'], sim_metric(preds, gts))
            _append_if_finite(scores['NSS'], nss_metric(preds, fix))
            _append_if_finite(scores['AUC-Borji'], auc_borji_metric(preds, fix))
            _append_if_finite(scores['AUC-J'], auc_judd_metric(preds, fix))
            _append_if_finite(scores['sAUC'], sauc_metric(preds, fix))
    return {k: (float(np.mean(v)) if len(v) else float('nan')) for k, v in scores.items()}

In [ ]:
print(f'Evaluating on test set ({len(test_ds)} samples)...')
print('Note: AUC-Borji + AUC-J + sAUC are evaluated on the test set.')
print('NOTE: expected values below are from the 50-epoch checkpoint (paper Table 2).')
print('      Running with tdsp_40epoch.pth — numbers may differ.')

final_metrics = evaluate_saliency_model(model, test_loader, device)

print()
print('='*45)
print(f"{'Metric':<15} {'Score':>8}  {'Expected':>10}")
print('-'*45)
expected = {'CC': 0.6423, 'KL': 0.9270, 'SIM': 0.5010, 'NSS': 3.4583, 'AUC-Borji': 0.9486}  # paper Table 2 (50-epoch)
for k, v in final_metrics.items():
    if k in expected:
        print(f'{k:<15} {v:>8.4f}  ({expected[k]:>8.4f})')
    else:
        print(f"{k:<15} {v:>8.4f}  ({'n/a':>8})")
print('='*45)

m = final_metrics
print()
print('LaTeX row for Table 3:')
print(f"TDSal (Ours) & {m['CC']:.4f} & {m['KL']:.4f} & {m['SIM']:.4f} & {m['NSS']:.4f} & {m['AUC-Borji']:.4f} & {m['AUC-J']:.4f} & {m['sAUC']:.4f} \\\\")


---
## Section D — Ablation Re-Evaluation (`AUC-Borji` sign fix rerun)


In [ ]:
from sentence_transformers import SentenceTransformer as _SBert

class _YOLOBackbone(nn.Module):
    def __init__(self):
        super().__init__()
        yolo = YOLO('yolov5su.pt')
        self.backbone = yolo.model.model[:10]
        for p in self.backbone.parameters(): p.requires_grad = False
        self.num_pre_fpm_feature_maps = 1
        self.num_multiscale_fusions_before_fpm = 0
    def get_pre_fpm_feature_info(self):
        return {
            'num_pre_fpm_feature_maps': self.num_pre_fpm_feature_maps,
            'num_multiscale_fusions_before_fpm': self.num_multiscale_fusions_before_fpm,
        }
    def forward(self, x): return self.backbone(x).clone()

class _FPM(nn.Module):
    def __init__(self, in_ch=512, out_ch=128):
        super().__init__()
        self.conv = nn.Conv2d(in_ch, out_ch, 1)
    def forward(self, x): return self.conv(x)

class _TaskEncoderSBERT(nn.Module):
    def __init__(self, out_dim=64):
        super().__init__()
        self.encoder = _SBert('all-MiniLM-L6-v2')
        for p in self.encoder.parameters(): p.requires_grad = False
        self.fc = nn.Linear(384, out_dim)
    def forward(self, texts, dev):
        emb = self.encoder.encode(texts, convert_to_tensor=True)
        return F.relu(self.fc(emb.detach().clone().to(dev)))

class _TaskEncoderID(nn.Module):
    def __init__(self, num_tasks=4, out_dim=64):
        super().__init__()
        self.emb = nn.Embedding(num_tasks, out_dim)
    def forward(self, ids, dev): return self.emb(ids.to(dev))

class _TFusion(nn.Module):
    def __init__(self, vis_dim=128, task_dim=64):
        super().__init__()
        self.proj = nn.Linear(task_dim, vis_dim)
        self.enc  = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d_model=vis_dim, nhead=4, batch_first=False),
            num_layers=1, enable_nested_tensor=False)
    def forward(self, v, t):
        B, C, H, W = v.shape
        v_s = rearrange(v, 'b c h w -> (h w) b c')
        t_q = rearrange(self.proj(t), 'b c -> 1 b c')
        x   = self.enc(torch.cat([t_q, v_s], 0))[1:]
        return rearrange(x, '(h w) b c -> b c h w', h=H, w=W)

class _FiLM(nn.Module):
    def __init__(self, vis_dim=128, task_dim=64):
        super().__init__()
        self.fc = nn.Linear(task_dim, 2 * vis_dim)
    def forward(self, v, t):
        B, C, _, _ = v.shape
        g, b = self.fc(t).chunk(2, dim=1)
        return g.view(B, C, 1, 1) * v + b.view(B, C, 1, 1)

class _Dec(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(ch, 64, 3, padding=1), nn.ReLU(),
            nn.ConvTranspose2d(64, 32, 4, 2, 1), nn.ReLU(),
            nn.ConvTranspose2d(32,  1, 4, 2, 1), nn.Sigmoid())
    def forward(self, x): return self.net(x)

class TDYSN(nn.Module):
    def __init__(self, use_task=True, use_transformer=True,
                 use_sbert=True, use_fpm=True, num_tasks=4):
        super().__init__()
        self.use_task  = use_task
        self.use_sbert = use_sbert
        self.backbone  = _YOLOBackbone()
        self.fpm       = _FPM() if use_fpm else nn.Identity()
        self.vis_dim   = 128 if use_fpm else 512
        self.task_enc  = _TaskEncoderSBERT() if use_sbert else _TaskEncoderID(num_tasks)
        self.fusion    = (_TFusion(self.vis_dim) if use_transformer
                          else _FiLM(self.vis_dim))
        self.decoder   = _Dec(self.vis_dim)
    def forward(self, imgs, task_desc=None, task_ids=None):
        v = self.fpm(self.backbone(imgs))
        if self.use_task:
            t = (self.task_enc(task_desc, imgs.device) if self.use_sbert
                 else self.task_enc(task_ids, imgs.device))
        else:
            t = torch.zeros(imgs.size(0), 64, device=imgs.device)
        return self.decoder(self.fusion(v, t))

In [ ]:
import pandas as pd

task_map  = {'count people': 0, 'detect the emotion': 1,
             'free view': 2, 'identify the action': 3}
num_tasks = 4

ablation_runs = [
    ('Full TDSal',      dict(use_task=True,  use_transformer=True,  use_sbert=True,  use_fpm=True),  'full.pth'),
    ('w/o Task',        dict(use_task=False, use_transformer=True,  use_sbert=True,  use_fpm=True),  'no_task.pth'),
    ('w/o Transformer', dict(use_task=True,  use_transformer=False, use_sbert=True,  use_fpm=True),  'no_transformer.pth'),
    ('w/o SBERT',       dict(use_task=True,  use_transformer=True,  use_sbert=False, use_fpm=True),  'no_sbert.pth'),
    ('w/o FPM',         dict(use_task=True,  use_transformer=True,  use_sbert=True,  use_fpm=False), 'no_fpm.pth'),
]

def eval_ablation(m, loader, dev):
    m.eval()
    scores = {k: [] for k in ['CC', 'KL', 'SIM', 'NSS', 'AUC-Borji', 'AUC-J', 'sAUC']}
    with torch.no_grad():
        for b in loader:
            imgs  = b['stimuli'].to(dev)
            gts   = b['fdm'].to(dev)
            tasks = b['task_description']
            t_ids = None
            if not getattr(m, 'use_sbert', True):
                t_ids = torch.tensor([task_map[t] for t in tasks], dtype=torch.long)
            preds = m(imgs, task_desc=tasks, task_ids=t_ids)
            preds = F.interpolate(preds, gts.shape[-2:], mode='bilinear', align_corners=False)
            fix   = (gts > 0.5).float()
            _append_if_finite(scores['CC'], cc_metric(preds, gts))
            _append_if_finite(scores['KL'], kl_metric(preds, gts))
            _append_if_finite(scores['SIM'], sim_metric(preds, gts))
            _append_if_finite(scores['NSS'], nss_metric(preds, fix))
            _append_if_finite(scores['AUC-Borji'], auc_borji_metric(preds, fix))
            _append_if_finite(scores['AUC-J'], auc_judd_metric(preds, fix))
            _append_if_finite(scores['sAUC'], sauc_metric(preds, fix))
    return {k: (float(np.mean(v)) if len(v) else float('nan')) for k, v in scores.items()}

rows = []
for name, cfg, fname in ablation_runs:
    path = os.path.join(ABLATION_BASE, fname)
    if not os.path.exists(path):
        print(f'SKIP (file not found): {path}')
        continue
    m = TDYSN(num_tasks=num_tasks, **cfg).to(device)
    m.load_state_dict(safe_torch_load(path, map_location=device), strict=False)
    res = eval_ablation(m, test_loader, device)
    res['Model'] = name
    rows.append(res)
    print(f'{name}: {res}')

if rows:
    df = pd.DataFrame(rows)[['Model','CC','KL','SIM','NSS','AUC-Borji','AUC-J','sAUC']].round(4)
    print('\n', df.to_string(index=False))
    print('\nLaTeX ablation table:')
    print(df.to_latex(
        index=False,
        caption=r'Ablation study (corrected AUC-Borji + added AUC-J and sAUC).',
        label='tab:ablation', column_format='lccccccc'
    ))
else:
    print('No ablation checkpoints found — Section D skipped.')

---
## Visualization — Qualitative Results on Test Set

In [ ]:
def visualize_test_sample(mdl, ds, full_ds, n_tasks=4, alpha=0.55, cmap='jet'):
    dev = next(mdl.parameters()).device
    for _ in range(100):
        rand_idx  = random.randint(0, len(ds) - 1)
        orig_idx  = ds.indices[rand_idx]
        _, fdm_path, _ = full_ds.samples[orig_idx]
        base = os.path.splitext(os.path.basename(fdm_path))[0]
        matched = [i for i, (_, fp, _) in enumerate(full_ds.samples)
                   if os.path.splitext(os.path.basename(fp))[0] == base]
        if len(matched) == n_tasks:
            break
    else:
        print('Could not find a sample with all 4 tasks; adjust n_tasks.')
        return
    samples = [full_ds[i] for i in matched]
    imgs_t  = torch.stack([s['stimuli'] for s in samples]).to(dev)
    fdm_t   = torch.stack([s['fdm'] for s in samples])
    descs   = [s['task_description'] for s in samples]
    with torch.no_grad():
        preds = mdl(imgs_t, descs)
        preds = F.interpolate(preds, fdm_t.shape[-2:], mode='bilinear', align_corners=False)
    img_np  = imgs_t.detach().cpu().permute(0, 2, 3, 1).numpy()
    pred_np = preds.detach().cpu().numpy()
    gt_np   = fdm_t.numpy()
    img_np = np.clip(img_np, 0, 1)
    pred_norm = pred_np[:, 0]
    pred_norm = (pred_norm - pred_norm.min(axis=(1, 2), keepdims=True)) / (
        pred_norm.max(axis=(1, 2), keepdims=True) - pred_norm.min(axis=(1, 2), keepdims=True) + 1e-8
    )
    gt_norm = gt_np[:, 0]
    gt_norm = (gt_norm - gt_norm.min(axis=(1, 2), keepdims=True)) / (
        gt_norm.max(axis=(1, 2), keepdims=True) - gt_norm.min(axis=(1, 2), keepdims=True) + 1e-8
    )
    cm = plt.get_cmap(cmap)
    fig, axes = plt.subplots(2, n_tasks, figsize=(4 * n_tasks, 7))
    for i in range(n_tasks):
        pred_rgb = cm(pred_norm[i])[..., :3]
        gt_rgb   = cm(gt_norm[i])[..., :3]
        pred_overlay = (1 - alpha) * img_np[i] + alpha * pred_rgb
        gt_overlay   = (1 - alpha) * img_np[i] + alpha * gt_rgb
        axes[0, i].imshow(np.clip(pred_overlay, 0, 1))
        axes[0, i].axis('off')
        axes[0, i].set_title(f'Prediction | {descs[i]}', fontsize=9)
        axes[1, i].imshow(np.clip(gt_overlay, 0, 1))
        axes[1, i].axis('off')
    axes[0, 0].set_ylabel('Pred Overlay', fontsize=10)
    axes[1, 0].set_ylabel('Ground Truth Overlay', fontsize=10)
    axes[0, -1].text(1.03, 0.5, 'Prediction', transform=axes[0, -1].transAxes,
                    rotation=90, va='center', ha='left', fontsize=10, fontweight='bold')
    axes[1, -1].text(1.03, 0.5, 'Ground Truth', transform=axes[1, -1].transAxes,
                    rotation=90, va='center', ha='left', fontsize=10, fontweight='bold')
    fig.suptitle(f'Test sample: {base} | Heatmap overlay (cmap={cmap}, alpha={alpha})', fontsize=11)
    plt.tight_layout()
    plt.show()
visualize_test_sample(model, test_ds, dataset)
